# importing libraries

In [120]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

from langchain_community.vectorstores import FAISS 
from langchain.agents import create_agent      


In [121]:
load_dotenv()

True

In [122]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("Environment Variables Loaded")

Environment Variables Loaded


# Loading our Data

In [123]:
DATA_PATH = os.path.join(os.getcwd(), "data", "hr_policy.txt")


# Data Injestion

In [124]:
loader = TextLoader(DATA_PATH, encoding="utf-8")

documents = loader.load()
print(documents)


[Document(metadata={'source': 'c:\\Users\\sahuk\\Desktop\\Projects\\AI_Engineer\\Gen_AI-Projects\\02_HR_Policy_RAG_Assistant\\data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll n

In [125]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [126]:
print(documents[0].metadata)

{'source': 'c:\\Users\\sahuk\\Desktop\\Projects\\AI_Engineer\\Gen_AI-Projects\\02_HR_Policy_RAG_Assistant\\data\\hr_policy.txt'}


In [127]:
print(f"Total characters in the document: {len(documents[0].page_content)}")

Total characters in the document: 2597


# Text Spilting

In [128]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'c:\\Users\\sahuk\\Desktop\\Projects\\AI_Engineer\\Gen_AI-Projects\\02_HR_Policy_RAG_Assistant\\data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'c:\\Users\\sahuk\\Desktop\\Projects\\AI_Engineer\\Gen_AI-Projects\\02_HR_Policy_RAG_Assistant\\data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'c:\\Users\\sahuk\\Desktop\\Projects\\AI_Engineer\\Gen_AI-Projects\\02_HR_Policy_RAG_Assistant\\data\\hr_poli

In [129]:
len(chunks)

9

# Embedding our Data

In [130]:

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


# Store data in Vector DB

In [131]:
vector_store = FAISS.from_documents(chunks , embeddings_model)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


# Testing

In [132]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



In [133]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def search_hr_policy(question: str) -> str:
    """Search the HR policy documents for information relevant to the query."""
    matching_chunks = retriever.invoke(question)
    return "\n\n" . join(chunk.page_content for chunk in matching_chunks)

# Data Retrival 

LLM

In [134]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5  # creativity 
)

llm.model_name

'openai/gpt-oss-120b'

In [135]:
tr = llm.invoke("Hey what is the leave policy")

In [136]:
tr.content

'Sure thing! Could you let me know which organization or country’s leave policy you’re interested in? That way I can give you the most relevant information—whether you need details on vacation days, sick leave, parental leave, or any other type of time‑off.'

# AI AGENT

In [137]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [138]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [139]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [142]:
print(response["messages"][-2].content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.


In [141]:
print(response["messages"][-1].content)

I’m the HR assistant for **Acme Corp**.
